In [56]:
%run 00_chargement_donnees.ipynb

# Indicateurs de production — Sillons

## Sillons — Volume mensuel par année

> ⚠️ **Note : filtre CLIENT sur 2024 non fiable**
> Pour l'année 2024, la correspondance OTP → Client est une estimation basée sur les numéros de train disponibles. La donnée client précise n'était pas renseignée à l'époque. Les résultats filtrés par client sur 2024 sont donc à interpréter avec prudence.

In [57]:
clients = ['TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SUD', 'CHAILLOUE', 'ETF DIRECTION MATERIEL', 'EXTERNE']

dropdown_annee = widgets.Dropdown(options=[2024, 2025, 2026], value=2025, description='Année :')
filtre_client = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def on_annee_change(change):
    if change['new'] == 2024:
        filtre_client.value = 'TOUS'
        filtre_client.disabled = True
    else:
        filtre_client.disabled = False

dropdown_annee.observe(on_annee_change, names='value')

def afficher_sillons(annee, client):
    df_f = df_all[df_all['ANNEE'] == annee].dropna(subset=['MOIS']).copy()
    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    vol = df_f.groupby('MOIS').size().reset_index(name='Nombre de sillons')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.bar(vol, x='PERIODE', y='Nombre de sillons',
                 title=f'Sillons circulés par mois — {annee}',
                 category_orders={'PERIODE': noms_mois},
                 text='Nombre de sillons')
    fig.update_traces(textposition='outside')
    fig.show()

widgets.interact(afficher_sillons, annee=dropdown_annee, client=filtre_client)

# Comparaison des 3 années sur une même courbe
filtre_client_courbe = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def afficher_courbe(client):
    df_f = df_all.dropna(subset=['MOIS']).copy()
    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    vol_courbe = df_f.groupby(['ANNEE', 'MOIS']).size().reset_index(name='Nombre de sillons')
    vol_courbe['PERIODE'] = vol_courbe['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.line(vol_courbe, x='PERIODE', y='Nombre de sillons',
                  color='ANNEE', markers=True,
                  title='Tendance des sillons circulés par mois — 2024 / 2025 / 2026',
                  category_orders={'PERIODE': noms_mois},
                  labels={'ANNEE': 'Année'})
    fig.show()

widgets.interact(afficher_courbe, client=filtre_client_courbe)

interactive(children=(Dropdown(description='Année :', index=1, options=(2024, 2025, 2026), value=2025), Dropdo…

interactive(children=(Dropdown(description='Client :', options=('TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SU…

<function __main__.afficher_courbe(client)>

## Sillons — Tendance mensuelle par client

In [58]:
filtre_annee_client = widgets.Dropdown(options=[2024, 2025, 2026], value=2025, description='Année :')

def afficher_courbe_par_client(annee):
    df_f = df_all[df_all['ANNEE'] == annee].dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    vol = df_f.groupby(['MOIS', 'CLIENT']).size().reset_index(name='Nombre de sillons')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.line(vol, x='PERIODE', y='Nombre de sillons',
                  color='CLIENT', markers=True,
                  title=f'Tendance des sillons par mois et par client — {annee}',
                  category_orders={'PERIODE': noms_mois},
                  labels={'CLIENT': 'Client'})
    fig.show()

widgets.interact(afficher_courbe_par_client, annee=filtre_annee_client)

interactive(children=(Dropdown(description='Année :', index=1, options=(2024, 2025, 2026), value=2025), Output…

<function __main__.afficher_courbe_par_client(annee)>

## Sillons — Comparaison mensuelle 2024 / 2025 / 2026

In [59]:
filtre_client_comp = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def afficher_comparaison_mensuelle(client):
    df_f = df_all.dropna(subset=['MOIS']).copy()
    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    if client != 'TOUS':
        df_f = df_f[df_f['ANNEE'] != 2024]
        display(widgets.HTML(
            '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:8px 12px;margin-bottom:8px;">'
            '⚠️ <strong>Données 2024 non disponibles par client.</strong> '
            'Pour l\'année 2024, l\'OTP n\'était renseigné qu\'en cas de non-conformité. '
            'Le filtre client sur 2024 n\'est donc pas fiable et n\'est pas affiché.'
            '</div>'
        ))
        annees_affichees = ['2025', '2026']
    else:
        annees_affichees = ['2024', '2025', '2026']

    vol = df_f.groupby(['ANNEE', 'MOIS']).size().reset_index(name='Nombre de sillons')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])
    vol['ANNEE'] = vol['ANNEE'].astype(str)

    fig = px.bar(vol, x='PERIODE', y='Nombre de sillons',
                 color='ANNEE', barmode='group',
                 title='Comparaison mensuelle ' + ' / '.join(annees_affichees) + (f' — {client}' if client != 'TOUS' else ''),
                 category_orders={'PERIODE': noms_mois, 'ANNEE': annees_affichees},
                 labels={'ANNEE': 'Année'},
                 text='Nombre de sillons')
    fig.update_traces(textposition='outside')
    fig.show()

widgets.interact(afficher_comparaison_mensuelle, client=filtre_client_comp)

interactive(children=(Dropdown(description='Client :', options=('TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SU…

<function __main__.afficher_comparaison_mensuelle(client)>

---

# Indicateurs de production — Prestations

## Prestations — Volume mensuel par année

In [60]:
filtre_client_prest = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def afficher_prestations(client):
    df_f = df_all[
        (df_all['ANNEE'] == 2026) &
        (df_all['Numéro de spot'].notna())
    ].dropna(subset=['MOIS']).copy()

    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]

    df_f['MOIS'] = df_f['MOIS'].astype(int)

    vol = df_f.groupby('MOIS')['Numéro de spot'].nunique().reset_index(name='Nombre de prestations')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.bar(vol, x='PERIODE', y='Nombre de prestations',
                 title='Prestations réalisées par mois — 2026' + (f' — {client}' if client != 'TOUS' else ''),
                 category_orders={'PERIODE': noms_mois},
                 text='Nombre de prestations')
    fig.update_traces(textposition='outside')
    fig.show()

widgets.interact(afficher_prestations, client=filtre_client_prest)

interactive(children=(Dropdown(description='Client :', options=('TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SU…

<function __main__.afficher_prestations(client)>

---

## Répartition par client — Sillons (%) et Prestations (%)

In [61]:
filtre_annee_cam_sillons = widgets.Dropdown(options=[2025, 2026], value=2026, description='Année :')

def afficher_camembert_sillons(annee):
    df_f = df_all[df_all['ANNEE'] == annee].copy()
    vol = df_f.groupby('CLIENT').size().reset_index(name='count')

    fig = px.pie(vol, names='CLIENT', values='count',
                 title=f'Répartition des sillons par client — {annee}')
    fig.update_traces(texttemplate='%{label}<br>%{percent:.1%} (%{value})')
    fig.show()

widgets.interact(afficher_camembert_sillons, annee=filtre_annee_cam_sillons)

interactive(children=(Dropdown(description='Année :', index=1, options=(2025, 2026), value=2026), Output()), _…

<function __main__.afficher_camembert_sillons(annee)>

In [62]:
df_prest = df_all[
    (df_all['ANNEE'] == 2026) &
    (df_all['Numéro de spot'].notna())
].copy()
vol_prest = df_prest.groupby('CLIENT')['Numéro de spot'].nunique().reset_index(name='count')

fig = px.pie(vol_prest, names='CLIENT', values='count',
             title='Répartition des prestations par client — 2026')
fig.update_traces(texttemplate='%{label}<br>%{percent:.1%} (%{value})')
fig.show()

---

## Sillons circulés — Part de non-conformités par mois

In [63]:
filtre_annee_nc = widgets.Dropdown(options=[2024, 2025, 2026], value=2026, description='Année :')
filtre_client_nc = widgets.Dropdown(options=clients, value='TOUS', description='Client :')
imputations_options = ['TOUS'] + sorted(df_all['CAUSE NC'].dropna().str.strip().str.upper().unique().tolist())
motifs_options = ['TOUS'] + sorted(df_all['SOUS-CAUSE NC'].dropna().str.strip().str.upper().unique().tolist())
filtre_imputation_nc = widgets.Dropdown(options=imputations_options, value='TOUS', description='Imputations :')
filtre_motif_nc = widgets.Dropdown(options=motifs_options, value='TOUS', description='Motifs :')

def on_annee_change_nc(change):
    if change['new'] == 2024:
        filtre_client_nc.value = 'TOUS'
        filtre_client_nc.disabled = True
    else:
        filtre_client_nc.disabled = False

filtre_annee_nc.observe(on_annee_change_nc, names='value')

def afficher_sillons_nc(annee, client, imputation, motif):
    df_f = df_all[df_all['ANNEE'] == annee].dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]

    df_sans = df_f[~(df_f['NC'].apply(lambda x: str(x).strip().upper() == 'X'))].copy()
    df_avec = df_f[df_f['NC'].apply(lambda x: str(x).strip().upper() == 'X')].copy()

    if imputation != 'TOUS':
        df_avec = df_avec[df_avec['CAUSE NC'].str.strip().str.upper() == imputation]
    if motif != 'TOUS':
        df_avec = df_avec[df_avec['SOUS-CAUSE NC'].str.strip().str.upper() == motif]

    df_sans['CATEGORIE'] = 'Sans NC'
    df_avec['CATEGORIE'] = 'Avec NC'
    df_combined = pd.concat([df_sans, df_avec])

    vol = df_combined.groupby(['MOIS', 'CATEGORIE']).size().reset_index(name='Nombre de sillons')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.bar(vol, x='PERIODE', y='Nombre de sillons',
                 color='CATEGORIE', barmode='stack',
                 title=f'Sillons circulés par mois — dont NC — {annee}' + (f' — {client}' if client != 'TOUS' else ''),
                 category_orders={'PERIODE': noms_mois, 'CATEGORIE': ['Sans NC', 'Avec NC']},
                 color_discrete_map={'Sans NC': '#4C9BE8', 'Avec NC': '#E8534C'},
                 text='Nombre de sillons')
    fig.update_traces(textposition='inside')
    fig.show()

widgets.interact(afficher_sillons_nc, annee=filtre_annee_nc, client=filtre_client_nc,
                 imputation=filtre_imputation_nc, motif=filtre_motif_nc)

interactive(children=(Dropdown(description='Année :', index=2, options=(2024, 2025, 2026), value=2026), Dropdo…

<function __main__.afficher_sillons_nc(annee, client, imputation, motif)>

## NC — Comparaison mensuelle 2024 / 2025 / 2026

In [64]:
filtre_client_nc_comp = widgets.Dropdown(options=clients, value='TOUS', description='Client :')
filtre_imputation_nc_comp = widgets.Dropdown(options=imputations_options, value='TOUS', description='Imputations :')
filtre_motif_nc_comp = widgets.Dropdown(options=motifs_options, value='TOUS', description='Motifs :')

def afficher_comparaison_nc(client, imputation, motif):
    df_f = df_all.dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    if client != 'TOUS':
        df_f = df_f[df_f['ANNEE'] != 2024]
        display(widgets.HTML(
            '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:8px 12px;margin-bottom:8px;">'
            '⚠️ <strong>Données 2024 non disponibles par client.</strong> Seules les années 2025 et 2026 sont affichées.'
            '</div>'
        ))
        df_f = df_f[df_f['CLIENT'] == client]

    df_nc = df_f[df_f['NC'].apply(lambda x: str(x).strip().upper() == 'X')]

    if imputation != 'TOUS':
        df_nc = df_nc[df_nc['CAUSE NC'].str.strip().str.upper() == imputation]
    if motif != 'TOUS':
        df_nc = df_nc[df_nc['SOUS-CAUSE NC'].str.strip().str.upper() == motif]

    vol = df_nc.groupby(['ANNEE', 'MOIS']).size().reset_index(name='Nombre de NC')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])
    vol['ANNEE'] = vol['ANNEE'].astype(str)

    annees = sorted(df_nc['ANNEE'].unique().astype(str).tolist())

    fig = px.bar(vol, x='PERIODE', y='Nombre de NC',
                 color='ANNEE', barmode='group',
                 title='Nombre de NC par mois — ' + ' / '.join(annees) + (f' — {client}' if client != 'TOUS' else ''),
                 category_orders={'PERIODE': noms_mois, 'ANNEE': annees},
                 labels={'ANNEE': 'Année'},
                 text='Nombre de NC')
    fig.update_traces(textposition='outside')
    fig.show()

widgets.interact(afficher_comparaison_nc, client=filtre_client_nc_comp,
                 imputation=filtre_imputation_nc_comp, motif=filtre_motif_nc_comp)

interactive(children=(Dropdown(description='Client :', options=('TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SU…

<function __main__.afficher_comparaison_nc(client, imputation, motif)>

## NC — Tendance mensuelle 2025 / 2026

In [65]:
filtre_client_nc_courbe = widgets.Dropdown(options=clients, value='TOUS', description='Client :')
filtre_imputation_nc_courbe = widgets.Dropdown(options=imputations_options, value='TOUS', description='Imputations :')
filtre_motif_nc_courbe = widgets.Dropdown(options=motifs_options, value='TOUS', description='Motifs :')

def afficher_tendance_nc(client, imputation, motif):
    df_f = df_all[df_all['ANNEE'].isin([2025, 2026])].dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]

    df_nc = df_f[df_f['NC'].apply(lambda x: str(x).strip().upper() == 'X')]

    if imputation != 'TOUS':
        df_nc = df_nc[df_nc['CAUSE NC'].str.strip().str.upper() == imputation]
    if motif != 'TOUS':
        df_nc = df_nc[df_nc['SOUS-CAUSE NC'].str.strip().str.upper() == motif]

    vol = df_nc.groupby(['ANNEE', 'MOIS']).size().reset_index(name='Nombre de NC')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])
    vol['ANNEE'] = vol['ANNEE'].astype(str)

    fig = px.line(vol, x='PERIODE', y='Nombre de NC',
                  color='ANNEE', markers=True,
                  title='Tendance des NC — 2025 / 2026' + (f' — {client}' if client != 'TOUS' else ''),
                  category_orders={'PERIODE': noms_mois, 'ANNEE': ['2025', '2026']},
                  labels={'ANNEE': 'Année'})
    fig.show()

widgets.interact(afficher_tendance_nc, client=filtre_client_nc_courbe,
                 imputation=filtre_imputation_nc_courbe, motif=filtre_motif_nc_courbe)

interactive(children=(Dropdown(description='Client :', options=('TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SU…

<function __main__.afficher_tendance_nc(client, imputation, motif)>

## NC — Répartition par Imputation et par Motif

In [66]:
import datetime

date_min = df_all['DATE'].dropna().min().date()
date_max = df_all['DATE'].dropna().max().date()

filtre_date_debut = widgets.DatePicker(description='Date début :', value=date_min)
filtre_date_fin = widgets.DatePicker(description='Date fin :', value=date_max)
filtre_client_cam_nc = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

btn_afficher = widgets.Button(description='Afficher', button_style='primary')
output_cam_nc = widgets.Output()

def afficher_cam_nc(b=None):
    output_cam_nc.clear_output()
    with output_cam_nc:
        debut = filtre_date_debut.value
        fin = filtre_date_fin.value
        client = filtre_client_cam_nc.value

        if debut is None or fin is None:
            display(widgets.HTML('<div style="color:red;">⚠️ Veuillez sélectionner une date de début et de fin.</div>'))
            return
        if debut > fin:
            display(widgets.HTML('<div style="color:red;">⚠️ La date de début doit être antérieure à la date de fin.</div>'))
            return

        df_f = df_all[
            (df_all['NC'].apply(lambda x: str(x).strip().upper() == 'X')) &
            (df_all['DATE'].dt.date >= debut) &
            (df_all['DATE'].dt.date <= fin)
        ].copy()

        if client != 'TOUS':
            df_f = df_f[df_f['CLIENT'] == client]

        if df_f.empty:
            display(widgets.HTML('<div style="color:orange;">Aucune NC trouvée sur cette période.</div>'))
            return

        # Camembert Imputations
        vol_imp = df_f.groupby('CAUSE NC').size().reset_index(name='count')
        fig_imp = px.pie(vol_imp, names='CAUSE NC', values='count',
                         title=f'NC par Imputation — {debut} au {fin}' + (f' — {client}' if client != 'TOUS' else ''))
        fig_imp.update_traces(texttemplate='%{label}<br>%{percent:.1%} (%{value})')
        fig_imp.show()

        # Tableau Motifs x Imputations
        pivot = df_f.pivot_table(
            index='SOUS-CAUSE NC', columns='CAUSE NC', aggfunc='size', fill_value=0
        )
        pivot.columns.name = None
        pivot.index.name = 'Motif'
        pivot['TOTAL'] = pivot.sum(axis=1)
        pivot['%'] = (pivot['TOTAL'] / pivot['TOTAL'].sum() * 100).round(1)
        pivot = pivot.sort_values('TOTAL', ascending=False)

        total_row = pivot.sum(numeric_only=True).to_frame().T
        total_row.index = ['TOTAL']
        total_row['%'] = 100.0

        tableau = pd.concat([pivot, total_row])

        # Colonnes entières sauf %
        cols_int = [c for c in tableau.columns if c != '%']
        tableau[cols_int] = tableau[cols_int].astype(int)

        display(widgets.HTML('<b>NC par Motif et Imputation</b>'))
        display(tableau)

btn_afficher.on_click(afficher_cam_nc)

display(widgets.VBox([
    widgets.HBox([filtre_date_debut, filtre_date_fin, filtre_client_cam_nc, btn_afficher]),
    output_cam_nc
]))

afficher_cam_nc()

## NC — Volume mensuel par client

In [ ]:
filtre_annee_nc_client = widgets.Dropdown(options=[2024, 2025, 2026], value=2026, description='Année :')

def afficher_nc_par_client(annee):
    df_f = df_all[
        (df_all['ANNEE'] == annee) &
        (df_all['NC'].apply(lambda x: str(x).strip().upper() == 'X'))
    ].dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    vol = df_f.groupby(['MOIS', 'CLIENT']).size().reset_index(name='Nombre de NC')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.bar(vol, x='PERIODE', y='Nombre de NC',
                 color='CLIENT', barmode='group',
                 title=f'NC par client et par mois — {annee}',
                 category_orders={'PERIODE': noms_mois},
                 labels={'CLIENT': 'Client'},
                 text='Nombre de NC')
    fig.update_traces(textposition='outside')
    fig.show()

widgets.interact(afficher_nc_par_client, annee=filtre_annee_nc_client)

interactive(children=(Dropdown(description='Année :', index=2, options=(2024, 2025, 2026), value=2026), Output…

<function __main__.afficher_nc_par_client(annee)>

---

## Sillons circulés — Part de sillons calés par mois

In [ ]:
filtre_annee_cale = widgets.Dropdown(options=[2024, 2025, 2026], value=2026, description='Année :')
filtre_client_cale = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def on_annee_change_cale(change):
    if change['new'] == 2024:
        filtre_client_cale.value = 'TOUS'
        filtre_client_cale.disabled = True
    else:
        filtre_client_cale.disabled = False

filtre_annee_cale.observe(on_annee_change_cale, names='value')

def afficher_sillons_cales(annee, client):
    df_f = df_all[df_all['ANNEE'] == annee].dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]

    df_f['CATEGORIE'] = df_f['CALE'].apply(lambda x: 'Calé' if str(x).strip().upper() == 'X' else 'Non calé')

    vol = df_f.groupby(['MOIS', 'CATEGORIE']).size().reset_index(name='Nombre de sillons')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.bar(vol, x='PERIODE', y='Nombre de sillons',
                 color='CATEGORIE', barmode='stack',
                 title=f'Sillons circulés — dont calés — {annee}' + (f' — {client}' if client != 'TOUS' else ''),
                 category_orders={'PERIODE': noms_mois, 'CATEGORIE': ['Non calé', 'Calé']},
                 color_discrete_map={'Non calé': '#4C9BE8', 'Calé': '#E8A44C'},
                 text='Nombre de sillons')
    fig.update_traces(selector={'name': 'Non calé'}, textposition='inside', insidetextanchor='start')
    fig.update_traces(selector={'name': 'Calé'}, textposition='outside')
    fig.show()

widgets.interact(afficher_sillons_cales, annee=filtre_annee_cale, client=filtre_client_cale)

interactive(children=(Dropdown(description='Année :', index=2, options=(2024, 2025, 2026), value=2026), Dropdo…

<function __main__.afficher_sillons_cales(annee, client)>

## Sillons calés — Tendance mensuelle 2024 / 2025 / 2026

In [70]:
filtre_client_cale_courbe = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def afficher_tendance_cales(client):
    df_f = df_all[
        df_all['CALE'].apply(lambda x: str(x).strip().upper() == 'X')
    ].dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    if client != 'TOUS':
        df_f = df_f[df_f['ANNEE'] != 2024]
        display(widgets.HTML(
            '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:8px 12px;margin-bottom:8px;">'
            '⚠️ <strong>Données 2024 non disponibles par client.</strong> Seules les années 2025 et 2026 sont affichées.'
            '</div>'
        ))
        df_f = df_f[df_f['CLIENT'] == client]

    vol = df_f.groupby(['ANNEE', 'MOIS']).size().reset_index(name='Nombre de sillons calés')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])
    vol['ANNEE'] = vol['ANNEE'].astype(str)

    fig = px.line(vol, x='PERIODE', y='Nombre de sillons calés',
                  color='ANNEE', markers=True,
                  title='Tendance des sillons calés par mois' + (f' — {client}' if client != 'TOUS' else ''),
                  category_orders={'PERIODE': noms_mois, 'ANNEE': ['2024', '2025', '2026']},
                  labels={'ANNEE': 'Année'})
    fig.show()

widgets.interact(afficher_tendance_cales, client=filtre_client_cale_courbe)

interactive(children=(Dropdown(description='Client :', options=('TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SU…

<function __main__.afficher_tendance_cales(client)>

---

## Acheminements ajoutés en opérationnel

In [ ]:
filtre_annee_ajout = widgets.Dropdown(options=[2024, 2025, 2026], value=2026, description='Année :')
filtre_client_ajout = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def on_annee_change_ajout(change):
    if change['new'] == 2024:
        filtre_client_ajout.value = 'TOUS'
        filtre_client_ajout.disabled = True
    else:
        filtre_client_ajout.disabled = False

filtre_annee_ajout.observe(on_annee_change_ajout, names='value')

def afficher_ajout_op(annee, client):
    df_f = df_all[df_all['ANNEE'] == annee].dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    if client != 'TOUS':
        df_f = df_f[df_f['CLIENT'] == client]

    df_f['CATEGORIE'] = df_f['AJOUT ACHE EN OP'].apply(
        lambda x: 'Ajouté en OP' if str(x).strip().upper() == 'X' else 'Non ajouté'
    )

    vol = df_f.groupby(['MOIS', 'CATEGORIE']).size().reset_index(name='Nombre')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.bar(vol, x='PERIODE', y='Nombre',
                 color='CATEGORIE', barmode='stack',
                 title=f'Acheminements ajoutés en opérationnel — {annee}' + (f' — {client}' if client != 'TOUS' else ''),
                 category_orders={'PERIODE': noms_mois, 'CATEGORIE': ['Non ajouté', 'Ajouté en OP']},
                 color_discrete_map={'Non ajouté': '#4C9BE8', 'Ajouté en OP': '#7BC67E'},
                 text='Nombre')
    fig.update_traces(selector={'name': 'Non ajouté'}, textposition='inside', insidetextanchor='start')
    fig.update_traces(selector={'name': 'Ajouté en OP'}, textposition='outside')
    fig.show()

widgets.interact(afficher_ajout_op, annee=filtre_annee_ajout, client=filtre_client_ajout)

interactive(children=(Dropdown(description='Année :', index=2, options=(2024, 2025, 2026), value=2026), Dropdo…

<function __main__.afficher_ajout_op(annee, client)>

In [ ]:
filtre_client_ajout_courbe = widgets.Dropdown(options=clients, value='TOUS', description='Client :')

def afficher_tendance_ajout(client):
    df_f = df_all[
        df_all['AJOUT ACHE EN OP'].apply(lambda x: str(x).strip().upper() == 'X')
    ].dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    if client != 'TOUS':
        df_f = df_f[df_f['ANNEE'] != 2024]
        display(widgets.HTML(
            '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:8px 12px;margin-bottom:8px;">'
            '⚠️ <strong>Données 2024 non disponibles par client.</strong> Seules les années 2025 et 2026 sont affichées.'
            '</div>'
        ))
        df_f = df_f[df_f['CLIENT'] == client]

    vol = df_f.groupby(['ANNEE', 'MOIS']).size().reset_index(name='Ajoutés en OP')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])
    vol['ANNEE'] = vol['ANNEE'].astype(str)

    fig = px.line(vol, x='PERIODE', y='Ajoutés en OP',
                  color='ANNEE', markers=True,
                  title='Tendance des acheminements ajoutés en OP' + (f' — {client}' if client != 'TOUS' else ''),
                  category_orders={'PERIODE': noms_mois, 'ANNEE': ['2024', '2025', '2026']},
                  labels={'ANNEE': 'Année'})
    fig.show()

widgets.interact(afficher_tendance_ajout, client=filtre_client_ajout_courbe)

interactive(children=(Dropdown(description='Client :', options=('TOUS', 'TX MECA NORD', 'GRAND PROJET', 'DR SU…

<function __main__.afficher_tendance_ajout(client)>

In [73]:
filtre_annee_ajout_cam = widgets.Dropdown(options=[2025, 2026], value=2026, description='Année :')

def afficher_cam_ajout(annee):
    df_f = df_all[
        (df_all['ANNEE'] == annee) &
        (df_all['AJOUT ACHE EN OP'].apply(lambda x: str(x).strip().upper() == 'X'))
    ].copy()

    vol = df_f.groupby('CLIENT').size().reset_index(name='count')

    fig = px.pie(vol, names='CLIENT', values='count',
                 title=f'Ajouts en OP par client — {annee}')
    fig.update_traces(texttemplate='%{label}<br>%{percent:.1%} (%{value})')
    fig.show()

widgets.interact(afficher_cam_ajout, annee=filtre_annee_ajout_cam)

interactive(children=(Dropdown(description='Année :', index=1, options=(2025, 2026), value=2026), Output()), _…

<function __main__.afficher_cam_ajout(annee)>

In [74]:
filtre_annee_ajout_client = widgets.Dropdown(options=[2025, 2026], value=2026, description='Année :')

def afficher_ajout_par_client(annee):
    df_f = df_all[
        (df_all['ANNEE'] == annee) &
        (df_all['AJOUT ACHE EN OP'].apply(lambda x: str(x).strip().upper() == 'X'))
    ].dropna(subset=['MOIS']).copy()
    df_f['MOIS'] = df_f['MOIS'].astype(int)

    vol = df_f.groupby(['MOIS', 'CLIENT']).size().reset_index(name='Ajoutés en OP')
    vol['PERIODE'] = vol['MOIS'].apply(lambda m: noms_mois[m - 1])

    fig = px.line(vol, x='PERIODE', y='Ajoutés en OP',
                  color='CLIENT', markers=True,
                  title=f'Ajouts en OP par client et par mois — {annee}',
                  category_orders={'PERIODE': noms_mois},
                  labels={'CLIENT': 'Client'})
    fig.show()

widgets.interact(afficher_ajout_par_client, annee=filtre_annee_ajout_client)

interactive(children=(Dropdown(description='Année :', index=1, options=(2025, 2026), value=2026), Output()), _…

<function __main__.afficher_ajout_par_client(annee)>